# Imports

In [1]:
import gymnasium as gym
from ale_py import ALEInterface
import cv2
import time
import json
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from collections import deque
from tqdm import tqdm

# Environment

In [2]:
ENVIRONMENT = "ALE/Pong-v5"  # 选择 ATARI 环境，Pong-v5 具有离散动作空间
RENDER_GAME_WINDOW = False  # 是否渲染游戏窗口，False 表示不渲染

env = gym.make(ENVIRONMENT, render_mode="rgb_array")  # 创建 gym 环境，使用 rgb_array 模式获取图像

# Replay Buffer

In [3]:
class ReplayBuffer:
    """
    回放缓冲区，用于存储经验数据。
    """
    def __init__(self, max_size: int, observation_shape, action_shape, seed: int | None = None):
        """
        初始化回放缓冲区。

        Parameters:
            max_size: 缓冲区最大容量。
            observation_shape: 观测空间形状。
            action_shape: 动作空间形状。
            seed: 随机数种子，用于可重复性。
        """
        self.max_size = max_size
        self.observation_shape = observation_shape
        self.action_shape = action_shape
        self.ptr = 0  # 指向当前写入位置的指针
        self.size = 0  # 缓冲区当前大小
        self.seed = seed
        self.rng = np.random.default_rng(seed)  # 初始化随机数生成器

        # 初始化存储观测、动作、奖励、下一个观测和完成标志的数组
        self.observations = np.zeros((max_size,) + observation_shape, dtype=np.float32)  # 假设 observation 是图像，使用 float32
        self.actions = np.zeros(max_size, dtype=np.int64)  # 假设 action 是整数
        self.rewards = np.zeros((max_size,), dtype=np.float32)
        self.next_observations = np.zeros((max_size,) + observation_shape, dtype=np.float32)  # 假设 observation 是图像
        self.dones = np.zeros((max_size,), dtype=np.float32)  # 使用 float32 为了与 loss 计算一致

    def add(self, current_observations: np.ndarray, actions: np.ndarray, rewards: np.ndarray, next_observations: np.ndarray, dones: np.ndarray) -> None:
        """
        向缓冲区添加一条经验数据。
        """
        self.observations[self.ptr] = current_observations
        self.actions[self.ptr] = actions
        self.rewards[self.ptr] = rewards[0]
        self.next_observations[self.ptr] = next_observations
        self.dones[self.ptr] = dones[0]
        self.ptr = (self.ptr + 1) % self.max_size  # 更新指针，循环覆盖旧数据
        self.size = min(self.size + 1, self.max_size)  # 更新缓冲区大小

    def sample(self, n_samples: int, replace: bool = True) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        """
        从缓冲区随机采样一批经验数据。
        """
        indices = self.rng.choice(self.size, size=n_samples, replace=replace)  # 随机选择索引
        return (
            self.observations[indices],
            self.actions[indices],
            self.rewards[indices],
            self.next_observations[indices],
            self.dones[indices],
        )

    def clear(self) -> None:
        """
        清空缓冲区。
        """
        self.ptr = 0
        self.size = 0

    def __getitem__(self, index: int | np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        """
        通过索引获取缓冲区中的数据。
        """
        return (
            self.observations[index],
            self.actions[index],
            self.rewards[index],
            self.next_observations[index],
            self.dones[index],
        )

    def __len__(self) -> int:
        """
        返回缓冲区当前大小。
        """
        return self.size


# Model

In [4]:
class DuelCNN(nn.Module):
    """
    Duel CNN 网络结构。采用的 Dueling Network Architecture，这种架构将 Q-network 分为价值和优势两个分支，从而更有效地估计 Q 值
    """
    def __init__(self, h, w, output_size):
        """
        初始化 Duel CNN 网络。

        Parameters:
            h: 输入图像高度。
            w: 输入图像宽度。
            output_size: 输出动作数量。
        """
        super(DuelCNN, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=4, out_channels=32, kernel_size=8, stride=4)  # 第一个卷积层
        self.bn1 = nn.BatchNorm2d(32)  # 批量归一化
        convw, convh = self.conv2d_size_calc(w, h, kernel_size=8, stride=4)  # 计算卷积后图像尺寸
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=4, stride=2)  # 第二个卷积层
        self.bn2 = nn.BatchNorm2d(64)  # 批量归一化
        convw, convh = self.conv2d_size_calc(convw, convh, kernel_size=4, stride=2)  # 计算卷积后图像尺寸
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, stride=1)  # 第三个卷积层
        self.bn3 = nn.BatchNorm2d(64)  # 批量归一化
        convw, convh = self.conv2d_size_calc(convw, convh, kernel_size=3, stride=1)  # 计算卷积后图像尺寸

        linear_input_size = convw * convh * 64  # 计算全连接层输入尺寸

        # 动作分支 （优势分支）
        self.Alinear1 = nn.Linear(in_features=linear_input_size, out_features=128)  # 第一个全连接层
        self.Alrelu = nn.LeakyReLU()  # LeakyReLU 激活函数 （允许小的负值通过，避免全连接层中的梯度消失、在价值估计任务中保留更多信息、对Q值估计的稳定性有帮助）
        self.Alinear2 = nn.Linear(in_features=128, out_features=output_size)  # 第二个全连接层

        # 状态值分支 （价值分支）
        self.Vlinear1 = nn.Linear(in_features=linear_input_size, out_features=128)  # 第一个全连接层
        self.Vlrelu = nn.LeakyReLU()  # LeakyReLU 激活函数
        self.Vlinear2 = nn.Linear(in_features=128, out_features=1)  # 第二个全连接层

    def conv2d_size_calc(self, w, h, kernel_size=5, stride=2):
        """
        计算卷积层输出图像尺寸。
        """
        next_w = (w - (kernel_size - 1) - 1) // stride + 1
        next_h = (h - (kernel_size - 1) - 1) // stride + 1
        return next_w, next_h

    def forward(self, x):
        """
        前向传播。
        """
        x = F.relu(self.bn1(self.conv1(x)))  # 第一个卷积层，批量归一化，ReLU 激活
        x = F.relu(self.bn2(self.conv2(x)))  # 第二个卷积层，批量归一化，ReLU 激活
        x = F.relu(self.bn3(self.conv3(x)))  # 第三个卷积层，批量归一化，ReLU 激活

        x = x.view(x.size(0), -1)  # 将卷积层输出展平

        Ax = self.Alrelu(self.Alinear1(x))  # 动作分支，第一个全连接层，LeakyReLU 激活
        Ax = self.Alinear2(Ax)  # 动作分支，第二个全连接层

        Vx = self.Vlrelu(self.Vlinear1(x))  # 状态值分支，第一个全连接层，LeakyReLU 激活
        Vx = self.Vlinear2(Vx)  # 状态值分支，第二个全连接层

        q = Vx + (Ax - Ax.mean(dim=1, keepdim=True))  # 计算 Q 值，使用 Duel DQN 结构，修正 mean 的维度以匹配 Ax，Ax是各动作的优势值

        return q

In [35]:
model = DuelCNN(h=80, w=64, output_size=6)
print(model)

DuelCNN(
  (conv1): Conv2d(4, 32, kernel_size=(8, 8), stride=(4, 4))
  (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv2d(32, 64, kernel_size=(4, 4), stride=(2, 2))
  (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1))
  (bn3): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (Alinear1): Linear(in_features=1536, out_features=128, bias=True)
  (Alrelu): LeakyReLU(negative_slope=0.01)
  (Alinear2): Linear(in_features=128, out_features=6, bias=True)
  (Vlinear1): Linear(in_features=1536, out_features=128, bias=True)
  (Vlrelu): LeakyReLU(negative_slope=0.01)
  (Vlinear2): Linear(in_features=128, out_features=1, bias=True)
)


# Sampler

In [5]:
class Sampler:
    """
    动作采样器，用于选择动作。
    """
    def __init__(self, epsilon: float, seed: int | None = None, greedy: bool = False):
        """
        初始化动作采样器。

        Parameters:
            epsilon: 随机选择动作的概率。
            seed: 随机数种子，用于可重复性。
            greedy: 是否贪婪选择动作（选择概率最高的动作）。
        """
        self.epsilon = epsilon
        self.greedy = greedy
        self.rng = np.random.default_rng(seed)  # 初始化随机数生成器

    def __call__(self, probabilities: np.ndarray) -> np.ndarray:
        """
        根据概率选择动作。
        """
        if self.rng.random() < self.epsilon:
            # 随机选择动作
            return np.array([self.rng.integers(0, probabilities.shape[1]) for _ in range(probabilities.shape[0])])
        else:
            # 根据模型输出选择动作
            if self.greedy:
                # 贪婪选择动作
                return np.argmax(probabilities, axis=1)
            else:
                # 注意：原始代码的 sampler 没有从模型分布中采样，而是直接 greedy 选择。
                # 如果要实现从模型分布中采样，需要使用 Categorical 分布。
                # 这里为了保持与原始代码功能一致，仍然使用 greedy 选择。
                return np.argmax(probabilities, axis=1)

# Play the game

In [6]:
def play_game(
    model: nn.Module,  # 修改为 nn.Module 类型
    buffer: ReplayBuffer,
    env: gym.Env,
    steps: int,
    sampler: Sampler,
    observations: np.ndarray | None = None,
    one_episode: bool = False
) -> np.ndarray:
    """
    使用模型 `model` 在环境 `env` 中进行 `steps` 步的游戏，并将经验存储在 `buffer` 中。
    """
    current_observations = observations
    if current_observations is None:
        # 如果没有提供初始观测，则重置环境
        current_observations, _ = env.reset()
        current_observations = agent_preprocess(current_observations)  # 预处理初始状态
        current_observations = np.stack((current_observations, current_observations, current_observations, current_observations))  # 堆叠初始帧

    for _ in range(steps):
        # 选择动作
        with torch.no_grad():
            # 将当前观测转换为 torch 张量，并添加到 batch 维度
            state_tensor = torch.tensor(current_observations, dtype=torch.float, device=DEVICE).unsqueeze(0)
            # 使用模型预测 Q 值
            q_values = model(state_tensor)
            # 将 Q 值移到 CPU 并转换为 numpy 数组
            probabilities = q_values.cpu().numpy()
        # 使用采样器选择动作
        actions = sampler(probabilities)  # Sampler 需要处理 batch 的 probabilities

        # 执行动作并获取下一个观测、奖励和完成标志
        next_observations, rewards, terminated, truncated, _ = env.step(actions[0])  # actions 是 batch，这里取第一个
        done = terminated or truncated
        # 预处理下一个观测
        next_observations_processed = agent_preprocess(next_observations)
        # 堆叠下一个观测
        next_observations_stacked = np.stack((next_observations_processed, current_observations[0], current_observations[1], current_observations[2]))

        # 将经验添加到回放缓冲区
        buffer.add(current_observations, actions[0], np.array([rewards]), next_observations_stacked, np.array([done], dtype=np.float32))  # done 需要是 np.ndarray

        # 更新当前观测
        current_observations = next_observations_stacked

        # 如果完成且 one_episode 为 True，则结束游戏
        if done and one_episode:
            break

    return current_observations

# Loss

In [7]:
def qq_loss(
    current_observation: torch.Tensor,
    action: torch.Tensor,
    reward: torch.Tensor,
    next_observation: torch.Tensor,
    done: torch.Tensor,
    model: nn.Module,
    target_model: nn.Module,
    gamma: float
) -> torch.Tensor:
    """
    计算 Double DQN 损失。改进目标值计算方式，解决 Q 值过估计问题
    """
    # print("current_observation shape:", current_observation.shape)
    # print("action shape:", action.shape)
    q_values_output = model(current_observation)
    # print("model(current_observation) shape:", q_values_output.shape)
    action_unsqueeze = action.unsqueeze(1)
    # print("action.unsqueeze(1) shape:", action_unsqueeze.shape)

    # 获取当前 Q 值
    current_q_values = model(current_observation).gather(1, action.unsqueeze(1)).squeeze(1)
    # 在线模型预测下一个状态的 Q 值
    next_q_values_online = model(next_observation)
    # 在线模型选择动作
    next_actions = torch.argmax(next_q_values_online, dim=1).unsqueeze(1)
    # 目标模型评估 Q 值
    next_q_values_target = target_model(next_observation).gather(1, next_actions).squeeze(1)
    # Double DQN 目标 Q 值
    target_q_value = reward + gamma * next_q_values_target * (1 - done)

    # 计算均方误差损失
    loss = F.mse_loss(current_q_values, target_q_value.detach())
    return loss

# Training

In [8]:
# 预处理函数
def agent_preprocess(image):
    """图像预处理函数。"""
    state_size_h = 210  # 硬编码原始状态尺寸，或者从 env 中获取，但为了独立性，这里选择硬编码
    state_size_w = 160
    crop_dim = [20, state_size_h, 0, state_size_w]  # 定义裁剪区域
    target_h = 80  # 目标图像高度
    target_w = 64  # 目标图像宽度

    frame = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)  # 将图像转换为灰度图
    frame = frame[crop_dim[0]:crop_dim[1], crop_dim[2]:crop_dim[3]]  # 裁剪图像
    frame = cv2.resize(frame, (target_w, target_h))  # 缩放图像
    frame = frame.reshape(target_w, target_h) / 255  # 归一化图像
    return frame

def adaptive_epsilon(epsilon):
    """自适应 epsilon 衰减。"""
    epsilon_decay = EPSILON_DECAY  # epsilon 衰减率
    epsilon_minimum = EPSILON_MINIMUM  # epsilon 最小值
    if epsilon > epsilon_minimum:
        epsilon *= epsilon_decay  # epsilon 衰减
    return epsilon

In [9]:
# 超参数和配置
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAVE_MODELS = True  # 是否保存模型
MODEL_PATH = r"C:\Users\lcf14\Desktop\RL\Pong"  # 模型保存路径
SAVE_MODEL_INTERVAL = 10  # 模型保存间隔（episode）
TRAIN_MODEL = True  # 是否训练模型
LOAD_MODEL_FROM_FILE = False  # 是否从文件加载模型
LOAD_FILE_EPISODE = 0  # 加载模型的 episode
BATCH_SIZE = 64  # 批大小
MAX_EPISODE = 900  # 最大 episode 数
MAX_STEP = 100000  # 最大步数
MAX_MEMORY_LEN = 50000  # 最大回放缓冲区长度
MIN_MEMORY_LEN = 40000  # 最小回放缓冲区长度
GAMMA = 0.97  # 折扣因子
ALPHA = 0.00025  # 学习率
EPSILON_DECAY = 0.99  # epsilon 衰减率
EPSILON_MINIMUM = 0.05  # epsilon 最小值
INITIAL_EPSILON = 1.0  # 初始 epsilon
TARGET_UPDATE_INTERVAL = 1  # 每 episode 更新 target model

In [10]:
def train(online_model, target_model, optimizer, replay_buffer, gamma, batch_size):
    """执行单个训练步骤。"""
    if len(replay_buffer) < MIN_MEMORY_LEN:
        return 0, 0  # 或者返回 None, None  # 如果回放缓冲区长度不足，则不进行训练

    sampled_data = replay_buffer.sample(batch_size)  # 从回放缓冲区采样一批数据
    s_batch, a_batch, r_batch, ns_batch, d_batch = sampled_data  # 解包采样数据
    s_batch_tensor = torch.tensor(s_batch, dtype=torch.float, device=DEVICE)  # 将观测转换为 torch 张量
    a_batch_tensor = torch.tensor(a_batch, dtype=torch.long, device=DEVICE)  # 将动作转换为 torch 张量
    r_batch_tensor = torch.tensor(r_batch, dtype=torch.float, device=DEVICE)  # 将奖励转换为 torch 张量
    ns_batch_tensor = torch.tensor(ns_batch, dtype=torch.float, device=DEVICE)  # 将下一个观测转换为 torch 张量
    d_batch_tensor = torch.tensor(d_batch, dtype=torch.float, device=DEVICE)  # 将完成标志转换为 torch 张量

    # 计算损失
    loss = qq_loss(s_batch_tensor, a_batch_tensor, r_batch_tensor, ns_batch_tensor, d_batch_tensor, online_model, target_model, gamma)

    optimizer.zero_grad()  # 清空梯度
    loss.backward()  # 反向传播
    optimizer.step()  # 更新模型参数

    max_q_val = 0  # 初始化为 0
    if len(replay_buffer) >= MIN_MEMORY_LEN:  # 只有当 buffer 足够大时才计算 max_q_val
        with torch.no_grad():
            q_values = online_model(s_batch_tensor)  # 计算 Q 值
            max_q_val = torch.max(q_values).item()  # 获取最大 Q 值

    return loss.item(), max_q_val  # 返回损失和最大 Q 值

In [11]:
# 初始化模型, ReplayBuffer, Sampler 和 Optimizer
online_model = DuelCNN(h=80, w=64, output_size=env.action_space.n).to(DEVICE)  # 创建在线模型，并将其移动到指定设备
target_model = DuelCNN(h=80, w=64, output_size=env.action_space.n).to(DEVICE)  # 创建目标模型，并将其移动到指定设备
target_model.load_state_dict(online_model.state_dict())  # 将在线模型的权重加载到目标模型
target_model.eval()  # 将目标模型设置为评估模式
optimizer = optim.Adam(online_model.parameters(), lr=ALPHA)  # 创建 Adam 优化器，用于更新在线模型参数
replay_buffer = ReplayBuffer(MAX_MEMORY_LEN, (4, 64, 80), (1,), seed=0)  # 创建回放缓冲区
sampler = Sampler(epsilon=INITIAL_EPSILON, seed=0)  # 创建采样器

epsilon = INITIAL_EPSILON  # 初始化 epsilon 在训练循环外部

if LOAD_MODEL_FROM_FILE:
    # 如果从文件加载模型
    online_model.load_state_dict(torch.load(MODEL_PATH + str(LOAD_FILE_EPISODE) + ".pkl"))  # 加载模型权重
    with open(MODEL_PATH + str(LOAD_FILE_EPISODE) + '.json') as outfile:
        param = json.load(outfile)  # 加载模型参数
        epsilon = param.get('epsilon')  # 获取 epsilon 值
    startEpisode = LOAD_FILE_EPISODE + 1  # 设置起始 episode
    sampler.epsilon = epsilon  # 更新采样器的 epsilon

else:
    startEpisode = 1  # 如果不加载模型，起始 episode 为 1

last_100_ep_reward = deque(maxlen=100)  # 创建一个双端队列，用于存储最近 100 个 episode 的奖励
total_step = 1  # 初始化总步数

for episode in tqdm(range(startEpisode, MAX_EPISODE + 1), desc='Episode', unit='episode', dynamic_ncols=True):
    # 遍历每个 episode
    state, _ = env.reset()  # 重置环境
    state_processed = agent_preprocess(state)  # 预处理初始状态
    state = np.stack((state_processed, state_processed, state_processed, state_processed))  # 堆叠初始状态

    total_max_q_val = 0  # 初始化总最大 Q 值
    total_reward = 0  # 初始化总奖励
    total_loss = 0  # 初始化总损失

    for step in range(MAX_STEP):
        # 遍历每个时间步
        if RENDER_GAME_WINDOW:
            env.render()  # 如果需要渲染游戏窗口，则渲染

        # 探索 or 利用
        sampler.epsilon = epsilon  # 设置采样器的 epsilon
        with torch.no_grad():
            state_tensor = torch.tensor(state, dtype=torch.float, device=DEVICE).unsqueeze(0)  # 将状态转换为 tensor，并添加 batch 维度
            q_values = online_model(state_tensor)  # 计算 Q 值
            probabilities = q_values.cpu().numpy()  # 将 Q 值移动到 CPU 并转换为 numpy 数组
        action = sampler(probabilities)[0]  # 使用采样器选择动作

        next_state, reward, terminated, truncated, _ = env.step(action)  # 执行动作，获取下一个状态、奖励、完成标志
        done = terminated or truncated
        next_state_processed = agent_preprocess(next_state)  # 预处理下一个状态
        next_state = np.stack((next_state_processed, state[0], state[1], state[2]))  # 堆叠下一个状态

        replay_buffer.add(state, np.array([action]), np.array([reward]), next_state, np.array([done], dtype=np.float32))  # 将经验添加到回放缓冲区

        state = next_state  # 更新状态

        if TRAIN_MODEL:
            # 如果需要训练模型
            loss_val, max_q_val = train(online_model, target_model, optimizer, replay_buffer, GAMMA, BATCH_SIZE)  # 训练在线模型
            total_loss += loss_val  # 累加损失
            total_max_q_val += max_q_val  # 累加最大 Q 值

        total_reward += reward  # 累加奖励
        total_step += 1  # 累加总步数
        if total_step % 1000 == 0:
            epsilon = adaptive_epsilon(epsilon)  # 自适应衰减 epsilon
            sampler.epsilon = epsilon  # 更新采样器的 epsilon

        if done:
            # 如果 episode 完成
            epsilonDict = {'epsilon': epsilon}  # 创建 epsilon 字典

            if SAVE_MODELS and episode % SAVE_MODEL_INTERVAL == 0:
                # 如果需要保存模型且满足保存间隔
                weightsPath = MODEL_PATH + str(episode) + '.pkl'  # 设置模型权重保存路径
                epsilonPath = MODEL_PATH + str(episode) + '.json'  # 设置 epsilon 参数保存路径
                torch.save(online_model.state_dict(), weightsPath)  # 保存模型权重
                with open(epsilonPath, 'w') as outfile:
                    json.dump(epsilonDict, outfile)  # 保存 epsilon 参数

            if TRAIN_MODEL and episode % TARGET_UPDATE_INTERVAL == 0:
                # 如果需要训练模型且满足目标模型更新间隔
                target_model.load_state_dict(online_model.state_dict())  # 更新目标模型权重

            last_100_ep_reward.append(total_reward)  # 将当前 episode 奖励添加到双端队列
            avg_max_q_val = total_max_q_val / (step + 1) if (step + 1) > 0 else 0  # 计算平均最大 Q 值

            outStr = "Episode:{} Reward:{:.2f} Loss:{:.2f} Last_100_Avg_Rew:{:.3f} Avg_Max_Q:{:.3f} Epsilon:{:.2f} Step:{} CStep:{}".format(
                episode, total_reward, total_loss, np.mean(last_100_ep_reward), avg_max_q_val, epsilon, step, total_step
            )  # 构建输出字符串
            tqdm.write(outStr)  # 输出训练信息

            if SAVE_MODELS:
                outputPath = MODEL_PATH + "out" + '.txt'  # 设置输出文件路径
                with open(outputPath, 'a') as outfile:
                    outfile.write(outStr + "\n")  # 将训练信息写入文件
            break

C:\Users\lcf14\AppData\Local\Temp\ipykernel_22732\1124135929.py:28: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  self.actions[self.ptr] = actions
Episode:   0%|                                                                    | 1/900 [00:01<18:27,  1.23s/episode]

Episode:1 Reward:-21.00 Loss:0.00 Last_100_Avg_Rew:-21.000 Avg_Max_Q:0.000 Epsilon:1.00 Step:820 CStep:822


Episode:   0%|▏                                                                   | 2/900 [00:03<26:39,  1.78s/episode]

Episode:2 Reward:-20.00 Loss:0.00 Last_100_Avg_Rew:-20.500 Avg_Max_Q:0.000 Epsilon:0.99 Step:880 CStep:1703


Episode:   0%|▏                                                                   | 3/900 [00:06<37:16,  2.49s/episode]

...

Episode:898 Reward:1.00 Loss:10.78 Last_100_Avg_Rew:-4.920 Avg_Max_Q:1.259 Epsilon:0.05 Step:2858 CStep:2207825


Episode: 100%|██████████████████████████████████████████████████████████████▉| 899/900 [16:08:42<01:03, 63.90s/episode]

Episode:899 Reward:-11.00 Loss:9.08 Last_100_Avg_Rew:-5.020 Avg_Max_Q:1.265 Epsilon:0.05 Step:2247 CStep:2210073


Episode: 100%|███████████████████████████████████████████████████████████████| 900/900 [16:09:55<00:00, 64.66s/episode]

Episode:900 Reward:-3.00 Loss:10.59 Last_100_Avg_Rew:-5.100 Avg_Max_Q:1.263 Epsilon:0.05 Step:2754 CStep:2212828


# Testing

In [29]:
def test_model(env, model, num_episodes=3, render=True, video_path=r"C:\Users\lcf14\Desktop\RL\Pong"):
    """测试模型并保存视频。"""
    os.makedirs(video_path, exist_ok=True)  # 确保视频保存目录存在

    for episode in range(num_episodes):
        # 遍历每个测试 episode
        state, _ = env.reset()  # 重置环境
        state_processed = agent_preprocess(state)  # 预处理初始状态
        state = np.stack((state_processed, state_processed, state_processed, state_processed))  # 堆叠初始状态
        total_reward = 0  # 初始化总奖励
        step_count = 0  # 初始化步数
        frames = []  # 存储帧列表

        while True:
            # 循环直到 episode 结束
            if render:
                frame = env.render()  # 获取渲染帧
                frames.append(frame)  # 将帧添加到列表中
                time.sleep(0.02)  # 暂停 0.02 秒，控制渲染速度

            with torch.no_grad():
                # 计算动作
                state_tensor = torch.tensor(state, dtype=torch.float, device=DEVICE).unsqueeze(0)  # 将状态转换为 tensor，并添加 batch 维度
                q_values = model(state_tensor)  # 计算 Q 值
                probabilities = q_values.cpu().numpy()  # 将 Q 值移动到 CPU 并转换为 numpy 数组
            action = np.argmax(probabilities)  # 选择具有最高 Q 值的动作

            next_state, reward, terminated, truncated, _ = env.step(action)  # 执行动作，获取下一个状态、奖励、完成标志
            done = terminated or truncated
            next_state_processed = agent_preprocess(next_state)  # 预处理下一个状态
            next_state = np.stack((next_state_processed, state[0], state[1], state[2]))  # 堆叠下一个状态

            total_reward += reward  # 累加奖励
            step_count += 1  # 累加步数
            state = next_state  # 更新状态

            if done:
                # 如果 episode 完成
                print(f"Test Episode: {episode+1}, Reward: {total_reward:.2f}, Steps: {step_count}")  # 输出测试结果
                video_filename = os.path.join(video_path, f"pong_test_episode_{episode+1}.mp4")  # 设置视频文件名
                iio.mimsave(video_filename, frames, fps=30)  # 保存为 mp4 视频，帧率为 30
                frames = []  # 清空帧列表
                break
    env.close()  # 关闭环境

In [34]:
import os
import imageio as iio

ENVIRONMENT = "ALE/Pong-v5"  # 设置环境名称
env = gym.make(ENVIRONMENT, render_mode="rgb_array")  # 创建环境

model_path = r"C:\Users\lcf14\Desktop\RL\Pong900.pkl"  # 设置模型路径
online_model.load_state_dict(torch.load(model_path))  # 加载模型权重
test_model(env, online_model, num_episodes=3, render=True, video_path=r"C:\Users\lcf14\Desktop\RL\Pong")  # 调用测试函数并保存视频

IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (160, 210) to (160, 224) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


Test Episode: 1, Reward: 10.00, Steps: 2401


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (160, 210) to (160, 224) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


Test Episode: 2, Reward: 8.00, Steps: 2693


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (160, 210) to (160, 224) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


Test Episode: 3, Reward: 9.00, Steps: 2570
